In [13]:
import pandas as pd

df=pd.read_csv("Superstore.csv", encoding="latin-1")
df.head()
df.shape
df.isnull().sum()
df.duplicated().sum()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quantity

In [28]:
import pandas as pd
from sqlalchemy import create_engine
# 0. Load the RAW file fresh (change the path to where your CSV really is)

df=pd.read_csv("Superstore.csv", encoding="latin-1")
# 1. Clean column names FIRST
df.columns = df.columns.str.strip().str.replace(" ", "_").str.replace("-", "_")
print(df.columns.tolist())

# 2. Duplicates
print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates()

# 3. Missing values
print(df.isnull().sum())

# 4. Fix date formats (note the underscores now)
df["Order_Date"] = pd.to_datetime(df["Order_Date"])
df["Ship_Date"] = pd.to_datetime(df["Ship_Date"])

# 5. Add useful columns
df["Order_Year"] = df["Order_Date"].dt.year
df["Order_Month"] = df["Order_Date"].dt.to_period("M").astype(str)
df["Profit_Margin"] = df["Profit"] / df["Sales"]

# 6. Save the cleaned file
df.to_csv("superstore_clean.csv", index=False)
print("Shape:", df.shape)

['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub_Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit']
Duplicates: 0
Row_ID           0
Order_ID         0
Order_Date       0
Ship_Date        0
Ship_Mode        0
Customer_ID      0
Customer_Name    0
Segment          0
Country          0
City             0
State            0
Postal_Code      0
Region           0
Product_ID       0
Category         0
Sub_Category     0
Product_Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64
Shape: (9994, 24)


In [30]:
username="postgres"
password="focus"
host="localhost"
port="5432"
database="Superstore"

engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

table_name="store"
df.to_sql(table_name,engine,if_exists="replace",index=False)
print(f"Data Successfully loaded '{table_name}' in database '{database}'.")

Data Successfully loaded 'store' in database 'Superstore'.


In [32]:
from sqlalchemy import text
import pandas as pd

print(pd.read_sql("SELECT COUNT(*) AS total_rows FROM store", engine))
print(pd.read_sql("SELECT * FROM store LIMIT 3", engine).columns.tolist())
df.columns = df.columns.str.lower()
df.to_sql("store", engine, if_exists="replace", index=False)

   total_rows
0        9994
['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub_Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Order_Year', 'Order_Month', 'Profit_Margin']


994